In [ ]:
import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch

path_to_pwd_file = ''

with open(path_to_pwd_file, 'r') as file:
    pwd = file.read().strip()

def connect(pwd: str,
            usr_nm: str = 'postgres', 
            host: str = 'localhost',
            port: str = '5432',
            db_nm: str = 'postgres'
           ) -> psycopg2.extensions.connection | None:
    conn = psycopg2.connect(
        host=host,
        database=db_nm,
        user=usr_nm,
        port=port,
        password=pwd
    )
    return conn
    
def create_db(new_db_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:

    if conn is None:
        conn = connect(pwd, db_nm='postgres')
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    if cur is None:
        cur = conn.cursor()
    cur.execute(f"CREATE DATABASE {new_db_nm};")

    conn.set_isolation_level(0)

    return conn, cur

conn, cur = create_db('stocks_db')
cur = conn.cursor()

df = pd.read_csv(r'all_stocks_5yr.csv)
df['date'] = pd.to_datetime(df['date'])

cur.execute("""
    CREATE TABLE IF NOT EXISTS stocks (
        date    DATE,
        open    FLOAT,
        high    FLOAT,
        low     FLOAT,
        close   FLOAT,
        volume  BIGINT,
        ticker  VARCHAR(10)
    );
""")
conn.commit()

records = [tuple(row) for row in df.values]

execute_batch(cur, """
    INSERT INTO stocks (date, open, high, low, close, volume, ticker)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", records, page_size=1000)

conn.commit()
print(f"Loaded {len(df)} rows successfully!")

## Step 1 — Data Import

In [ ]:
def calculate_returns(conn: psycopg2.extensions.connection,
                      cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks 
        ADD COLUMN IF NOT EXISTS daily_return FLOAT,
        ADD COLUMN IF NOT EXISTS log_return FLOAT;
        
        UPDATE stocks s
        SET 
            daily_return = (s.close - prev.prev_close) / prev.prev_close,
            log_return   = LN(s.close / prev.prev_close)
        FROM (
            SELECT ticker, date,
                   LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS prev_close
            FROM stocks
        ) prev
        WHERE s.ticker = prev.ticker 
          AND s.date   = prev.date
          AND prev.prev_close IS NOT NULL;
    """)
    conn.commit()
    print('Returns calculated successfully!')

calculate_returns(conn, cur)

## Step 2 — Daily & Log Returns

In [ ]:
def calculate_rolling_stats(conn: psycopg2.extensions.connection,
                             cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks
        ADD COLUMN IF NOT EXISTS rolling_mean_7   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_7    FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_7    FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_7    FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_14  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_14   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_14   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_14   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_30  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_30   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_30   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_30   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_60  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_60   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_60   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_60   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_90  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_90   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_90   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_90   FLOAT;

        UPDATE stocks s
        SET
            -- 7-day
            rolling_mean_7  = r.mean_7,
            rolling_std_7   = r.std_7,
            rolling_min_7   = r.min_7,
            rolling_max_7   = r.max_7,
            -- 14-day
            rolling_mean_14 = r.mean_14,
            rolling_std_14  = r.std_14,
            rolling_min_14  = r.min_14,
            rolling_max_14  = r.max_14,
            -- 30-day
            rolling_mean_30 = r.mean_30,
            rolling_std_30  = r.std_30,
            rolling_min_30  = r.min_30,
            rolling_max_30  = r.max_30,
            -- 60-day
            rolling_mean_60 = r.mean_60,
            rolling_std_60  = r.std_60,
            rolling_min_60  = r.min_60,
            rolling_max_60  = r.max_60,
            -- 90-day
            rolling_mean_90 = r.mean_90,
            rolling_std_90  = r.std_90,
            rolling_min_90  = r.min_90,
            rolling_max_90  = r.max_90
        FROM (
            SELECT ticker, date,
                -- 7-day window
                AVG(daily_return)  OVER w7 AS mean_7,
                STDDEV(daily_return) OVER w7 AS std_7,
                MIN(daily_return)  OVER w7 AS min_7,
                MAX(daily_return)  OVER w7 AS max_7,
                -- 14-day window
                AVG(daily_return)  OVER w14 AS mean_14,
                STDDEV(daily_return) OVER w14 AS std_14,
                MIN(daily_return)  OVER w14 AS min_14,
                MAX(daily_return)  OVER w14 AS max_14,
                -- 30-day window
                AVG(daily_return)  OVER w30 AS mean_30,
                STDDEV(daily_return) OVER w30 AS std_30,
                MIN(daily_return)  OVER w30 AS min_30,
                MAX(daily_return)  OVER w30 AS max_30,
                -- 60-day window
                AVG(daily_return)  OVER w60 AS mean_60,
                STDDEV(daily_return) OVER w60 AS std_60,
                MIN(daily_return)  OVER w60 AS min_60,
                MAX(daily_return)  OVER w60 AS max_60,
                -- 90-day window
                AVG(daily_return)  OVER w90 AS mean_90,
                STDDEV(daily_return) OVER w90 AS std_90,
                MIN(daily_return)  OVER w90 AS min_90,
                MAX(daily_return)  OVER w90 AS max_90
            FROM stocks
            WINDOW
                w7  AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6  PRECEDING AND CURRENT ROW),
                w14 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
                w30 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                w60 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                w90 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW)
        ) r
        WHERE s.ticker = r.ticker
          AND s.date   = r.date;
    """)
    conn.commit()
    print('Rolling statistics calculated successfully!')

calculate_rolling_stats(conn, cur)

## Step 3 — Rolling Statistics

In [ ]:
def calculate_volatility_regimes(conn: psycopg2.extensions.connection,
                                  cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks
        ADD COLUMN IF NOT EXISTS volatility_regime VARCHAR(10);

        UPDATE stocks s
        SET volatility_regime = CASE
            WHEN r.rolling_std_30 < 0.01   THEN 'Low'
            WHEN r.rolling_std_30 < 0.02   THEN 'Normal'
            WHEN r.rolling_std_30 < 0.035  THEN 'High'
            ELSE                                 'Extreme'
        END
        FROM stocks r
        WHERE s.ticker = r.ticker
          AND s.date   = r.date
          AND r.rolling_std_30 IS NOT NULL;
    """)
    conn.commit()
    print('Volatility regimes classified successfully!')

calculate_volatility_regimes(conn, cur)

## Step 4 — Volatility Regime Classification

In [ ]:
def calculate_cumulative_returns(conn: psycopg2.extensions.connection,
                                  cur: psycopg2.extensions.cursor,
                                  start_date: str,
                                  end_date: str) -> pd.DataFrame:
    cur.execute("""
        SELECT ticker,
               EXP(SUM(LN(1 + daily_return))) - 1                    AS cumulative_return,
               POWER(EXP(SUM(LN(1 + daily_return))), 252.0 / COUNT(*)) - 1  AS annualized_return
        FROM stocks
        WHERE date BETWEEN %s AND %s
          AND daily_return IS NOT NULL
        GROUP BY ticker
        ORDER BY annualized_return DESC;
    """, (start_date, end_date))
    
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=['ticker', 'cumulative_return', 'annualized_return'])
    #print(df)
    return df

calculate_cumulative_returns(conn, cur, '2007-03-05', '2016-08-30')

## Step 5 — Cumulative & Annualized Returns

In [ ]:
def calculate_risk_metrics(conn: psycopg2.extensions.connection,
                            cur: psycopg2.extensions.cursor) -> pd.DataFrame:
    cur.execute("""
        WITH daily AS (
            SELECT ticker,
                   date,
                   daily_return,
                   -- running cumulative return to track equity curve
                   EXP(SUM(LN(1 + daily_return)) 
                       OVER (PARTITION BY ticker ORDER BY date)) AS equity_curve
            FROM stocks
            WHERE daily_return IS NOT NULL
        ),
        drawdown AS (
            SELECT ticker,
                   date,
                   daily_return,
                   equity_curve,
                   -- peak so far for this ticker
                   MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS peak,
                   -- drawdown at each point
                   (equity_curve - MAX(equity_curve) 
                       OVER (PARTITION BY ticker ORDER BY date)) 
                   / MAX(equity_curve) 
                       OVER (PARTITION BY ticker ORDER BY date) AS drawdown
            FROM daily
        )
        SELECT 
            ticker,
            -- annualized return
            (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1) AS annualized_return,
            -- annualized volatility
            STDDEV(daily_return) * SQRT(252) AS annualized_volatility,
            -- sharpe ratio (assuming 0 risk-free rate for simplicity)
            (AVG(daily_return) * 252) / (STDDEV(daily_return) * SQRT(252)) AS sharpe_ratio,
            -- max drawdown
            MIN(drawdown) AS max_drawdown,
            -- calmar ratio
            (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1) 
            / ABS(MIN(drawdown)) AS calmar_ratio
        FROM drawdown
        GROUP BY ticker
        ORDER BY sharpe_ratio DESC;
    """)
    
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=[
        'ticker', 'annualized_return', 'annualized_volatility',
        'sharpe_ratio', 'max_drawdown', 'calmar_ratio'
    ])
    print(df)
    return df

calculate_risk_metrics(conn, cur)

## Step 6 — Risk Metrics (Sharpe, Max Drawdown, Calmar)

In [ ]:
def build_final_table(conn: psycopg2.extensions.connection,
                      cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS stocks_final AS
        SELECT
            s.ticker,
            s.date,
            s.open,
            s.high,
            s.low,
            s.close,
            s.volume,
            -- returns
            s.daily_return,
            s.log_return,
            -- rolling stats
            s.rolling_mean_7,   s.rolling_std_7,   s.rolling_min_7,   s.rolling_max_7,
            s.rolling_mean_14,  s.rolling_std_14,  s.rolling_min_14,  s.rolling_max_14,
            s.rolling_mean_30,  s.rolling_std_30,  s.rolling_min_30,  s.rolling_max_30,
            s.rolling_mean_60,  s.rolling_std_60,  s.rolling_min_60,  s.rolling_max_60,
            s.rolling_mean_90,  s.rolling_std_90,  s.rolling_min_90,  s.rolling_max_90,
            -- regime
            s.volatility_regime,
            -- risk metrics joined in
            r.annualized_return,
            r.annualized_volatility,
            r.sharpe_ratio,
            r.max_drawdown,
            r.calmar_ratio
        FROM stocks s
        LEFT JOIN (
            WITH daily AS (
                SELECT ticker, date, daily_return,
                       EXP(SUM(LN(1 + daily_return)) 
                           OVER (PARTITION BY ticker ORDER BY date)) AS equity_curve
                FROM stocks WHERE daily_return IS NOT NULL
            ),
            drawdown AS (
                SELECT ticker, date, daily_return, equity_curve,
                       MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS peak,
                       (equity_curve - MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date))
                       / MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS drawdown
                FROM daily
            )
            SELECT ticker,
                   (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1)              AS annualized_return,
                   STDDEV(daily_return) * SQRT(252)                               AS annualized_volatility,
                   (AVG(daily_return) * 252) / (STDDEV(daily_return) * SQRT(252)) AS sharpe_ratio,
                   MIN(drawdown)                                                   AS max_drawdown,
                   (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1) 
                   / ABS(MIN(drawdown))                                           AS calmar_ratio
            FROM drawdown
            GROUP BY ticker
        ) r ON s.ticker = r.ticker
        ORDER BY s.ticker, s.date;

        CREATE INDEX IF NOT EXISTS idx_final_ticker_date 
        ON stocks_final (ticker, date);
    """)
    conn.commit()
    print('Final table built successfully!')

build_final_table(conn,cur)

## Step 7 — Final Consolidated Table

In [ ]:
def export_final_table(conn, cur, export_path: str) -> pd.DataFrame:
    cur.execute("SELECT * FROM stocks_final;")
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(export_path, index=False)
    print(f'Exported {len(df)} rows to {export_path}')
    return df
cur.execute("DROP TABLE IF EXISTS stocks_final;")
conn.commit()
build_final_table(conn, cur)
df_final = export_final_table(conn, cur, r'C:\Users\timur\Desktop\Python_projects_2026\stocks_final.csv')

## Export

In [ ]:
# check the shape
print(df_final.shape)

# check no weird nulls in key columns
print(df_final[['ticker', 'date', 'daily_return', 'log_return', 
                'rolling_std_30', 'volatility_regime', 
                'sharpe_ratio', 'max_drawdown']].isnull().sum())

# peek at a single stock
print(df_final[df_final['ticker'] == 'AAPL'].head(10))

# regime distribution - how many days in each regime across all stocks
print(df_final['volatility_regime'].value_counts())

# top 10 stocks by sharpe ratio
print(df_final.drop_duplicates('ticker')[['ticker', 'sharpe_ratio', 'annualized_return', 'max_drawdown']]
      .sort_values('sharpe_ratio', ascending=False)
      .head(10))

## Top 10 Stocks by Sharpe Ratio

DXC has the best risk-adjusted return at 2.19 — it earned 62% annually while keeping drawdown to only 8.1%. NVDA had the highest raw return (82% annualized) but only ranks 4th in Sharpe because it carried much higher volatility and a 25% max drawdown. This illustrates exactly why Sharpe ratio matters: raw return alone doesn't tell the full story.

In [57]:
print(df_final['volatility_regime'].value_counts(normalize=True).mul(100).round(1))

volatility_regime
Normal     52.7
Low        33.3
High       12.1
Extreme     1.9
Name: proportion, dtype: float64


## Volatility Regime Distribution

Over 86% of all stock-days across the 2013–2018 period fell in Low or Normal volatility regimes, consistent with the sustained bull market environment. Only 1.9% of days were classified as Extreme — roughly 12,000 stock-days — showing the metric does capture real volatility events even in a calm period.